In [0]:
# =========================================
# CONFIG
# =========================================

storage_account = "stspmobilitydev001"
container = "bronze"

account_fqdn = f"{storage_account}.dfs.core.windows.net"

# =========================================
# SECRETS
# =========================================

scope = "kv-sp-mobility"

client_id = dbutils.secrets.get(scope=scope, key="databricks-sp-client-id")
client_secret = dbutils.secrets.get(scope=scope, key="databricks-sp-secret")
tenant_id = dbutils.secrets.get(scope=scope, key="databricks-sp-tenant-id")

# =========================================
# LIMPA CONFIG ANTIGA (MUITO IMPORTANTE)
# =========================================

try:
    spark.conf.unset(f"fs.azure.account.key.{account_fqdn}")
except:
    pass

hc = spark._jsc.hadoopConfiguration()
hc.unset(f"fs.azure.account.key.{account_fqdn}")

# =========================================
# OAUTH CONFIG
# =========================================

configs = {
    f"fs.azure.account.auth.type.{account_fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{account_fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{account_fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{account_fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

for k, v in configs.items():
    spark.conf.set(k, v)
    hc.set(k, v)

# =========================================
# TESTE (CRÍTICO)
# =========================================

print("OAuth configurado para:", account_fqdn)

display(
    dbutils.fs.ls(f"abfss://{container}@{account_fqdn}/")
)